# RDFLib + SHACL 最小サンプル

この notebook では次を行います。

1. Turtle 形式の RDF graph を Python 文字列として定義する
2. `rdflib` で読み込む
3. triples を pandas DataFrame にする
4. SHACL shape を定義する
5. `pyshacl` で validation を実行する

題材はシンプルな **FireDoor / AlarmZone** です。
要件は次の通りです。

- FireDoor は `connectedTo` を少なくとも1つ持つべき
- FireDoor は `hasIdentifier` をちょうど1つ持つべき

最初はわざと不備のあるデータを入れて、SHACL で violation を確認します。

- RDF = データモデル
- Turtle / RDF/XML / JSON-LD = RDF の書き方

## 0. 必要ライブラリ

- `rdflib`
- `pandas`
- `pyshacl`

`pyshacl` が未導入なら次のセルを使ってください。

In [1]:
# 必要なら実行
%pip install pyshacl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 4.5 MB/s eta 0:00:00


In [2]:
from rdflib import Graph
import pandas as pd

try:
    from pyshacl import validate
except ImportError as e:
    raise ImportError(
        "pyshacl が見つかりません。上のセルの `%pip install pyshacl` を実行してください。"
    ) from e

## 1. サンプル RDF データ

ここでは `FD001` を FireDoor とし、わざと `connectedTo` を欠落させています。  
さらに `FD002` には `hasIdentifier` を2つ入れて、件数違反も作ります。

In [3]:
# .ttl は Turtle 形式のファイル拡張子
# 全部で8行のtriple
data_ttl = '''
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:FireDoor rdf:type rdf:Property .

ex:FD001 rdf:type ex:FireDoor ;
         ex:hasIdentifier "FD001" .

ex:FD002 rdf:type ex:FireDoor ;
         ex:hasIdentifier "FD002-A" ;
         ex:hasIdentifier "FD002-B" ;
         ex:connectedTo ex:ZoneA .

ex:ZoneA rdf:type ex:AlarmZone .
'''

## 2. RDF graph を読み込む

In [4]:
g = Graph()
g.parse(data=data_ttl, format='turtle')

print(f'Number of triples: {len(g)}')

Number of triples: 8


## 3. triples を表形式で確認する

In [5]:
rows = [(str(s), str(p), str(o)) for s, p, o in g]
df = pd.DataFrame(rows, columns=['subject', 'predicate', 'object'])
df.sort_values(['subject', 'predicate', 'object']).reset_index(drop=True)

,subject,predicate,object
0,http://example.org/FD001,http://example.org/hasIdentifier,FD001
1,http://example.org/FD001,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor
2,http://example.org/FD002,http://example.org/connectedTo,http://example.org/ZoneA
3,http://example.org/FD002,http://example.org/hasIdentifier,FD002-A
4,http://example.org/FD002,http://example.org/hasIdentifier,FD002-B
5,http://example.org/FD002,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/FireDoor
6,http://example.org/FireDoor,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...
7,http://example.org/ZoneA,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/AlarmZone


## 4. SHACL shape を定義する

今回は `FireDoorShape` を定義して、次を検査します。

- `hasIdentifier` は **1件以上 1件以下**
- `connectedTo` は **1件以上**
- `connectedTo` の先は `AlarmZone` 型

In [6]:
# SHACL を Turtle で書いたもの

# ex:FireDoor
# は本当は
# http://example.org/FireDoor
#
# 同様に
# sh:NodeShape
# http://www.w3.org/ns/shacl#NodeShape

# ex:FireDoorShape a sh:NodeShape ;
# これは
# ex:FireDoorShape rdf:type sh:NodeShape
# の省略形
# FireDoorShape は NodeShape 型である
# の意味

# Turtle では
# ; = 主語をそのまま維持して、述語を続ける
# . = この主語の記述は終わり

# ex:FireDoorShape
#     a sh:NodeShape ;
#     sh:targetClass ex:FireDoor ;
# は、主語 ex:FireDoorShape に対して
# a sh:NodeShape
# sh:targetClass ex:FireDoor
# の2つを書いています。

# sh:targetClass ex:FireDoor
# これは
# FireDoor クラスのノードを対象にする
# という意味

# sh:property [
#     sh:path ex:hasIdentifier ;
#     sh:minCount 1 ;
#     sh:maxCount 1 ;
#     sh:datatype xsd:string ;
# ] ;
# これは
# 匿名の property shape を1個作って、それを FireDoorShape にぶら下げている
# という意味です。
# FireDoor は hasIdentifier をちょうど1つ持ち、その値は文字列でなければならない
# という文。

# この FireDoorShape 全体は、要するにこうです。

# FireDoor に対する SHACL 検査ルールを定義する。
# FireDoor は:
# hasIdentifier をちょうど1つ持つこと
# その identifier は文字列であること
# connectedTo を少なくとも1つ持つこと
# その接続先は AlarmZone であること

shapes_ttl = '''
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:FireDoorShape
    a sh:NodeShape ;
    sh:targetClass ex:FireDoor ;
    sh:property [
        sh:path ex:hasIdentifier ;
        sh:minCount 1 ;
        sh:maxCount 1 ;
        sh:datatype xsd:string ;
    ] ;
    sh:property [
        sh:path ex:connectedTo ;
        sh:minCount 1 ;
        sh:class ex:AlarmZone ;
    ] .
'''

## 5. SHACL validation を実行する

In [7]:
conforms, results_graph, results_text = validate(
    data_graph=g,
    shacl_graph=Graph().parse(data=shapes_ttl, format='turtle'),
    inference='rdfs',
    abort_on_first=False,
    allow_infos=True,
    allow_warnings=True,
)

print('Conforms:', conforms)
print(results_text)

Conforms: False
Validation Report
Conforms: False
Results (2):
Constraint Violation in MaxCountConstraintComponent (http://www.w3.org/ns/shacl#MaxCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:hasIdentifier ]
	Focus Node: ex:FD002
	Result Path: ex:hasIdentifier
	Message: More than 1 values on ex:FD002->ex:hasIdentifier
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class ex:AlarmZone ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:connectedTo ]
	Focus Node: ex:FD001
	Result Path: ex:connectedTo
	Message: Less than 1 values on ex:FD001->ex:connectedTo



## 6. Validation 結果を RDF graph として眺める

`results_graph` も RDF なので、必要なら triples 化して後続分析に回せます。

In [8]:
result_rows = [(str(s), str(p), str(o)) for s, p, o in results_graph]
result_df = pd.DataFrame(result_rows, columns=['subject', 'predicate', 'object'])
result_df.sort_values(['subject', 'predicate', 'object']).reset_index(drop=True).head(30)

,subject,predicate,object
0,N3bb1c47d105347b5b7dc71696f029924,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/ns/shacl#ValidationResult
1,N3bb1c47d105347b5b7dc71696f029924,http://www.w3.org/ns/shacl#focusNode,http://example.org/FD002
2,N3bb1c47d105347b5b7dc71696f029924,http://www.w3.org/ns/shacl#resultMessage,More than 1 values on ex:FD002->ex:hasIdentifier
3,N3bb1c47d105347b5b7dc71696f029924,http://www.w3.org/ns/shacl#resultPath,http://example.org/hasIdentifier
4,N3bb1c47d105347b5b7dc71696f029924,http://www.w3.org/ns/shacl#resultSeverity,http://www.w3.org/ns/shacl#Violation
5,N3bb1c47d105347b5b7dc71696f029924,http://www.w3.org/ns/shacl#sourceConstraintCom...,http://www.w3.org/ns/shacl#MaxCountConstraintC...
6,N3bb1c47d105347b5b7dc71696f029924,http://www.w3.org/ns/shacl#sourceShape,na399e143a21e460d978cd7037adcea67b1
7,N8a727ce1e93a4401972065581e4eec7f,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/ns/shacl#ValidationReport
8,N8a727ce1e93a4401972065581e4eec7f,http://www.w3.org/ns/shacl#conforms,false
9,N8a727ce1e93a4401972065581e4eec7f,http://www.w3.org/ns/shacl#result,N3bb1c47d105347b5b7dc71696f029924


## 7. データを修正して再検証する

今度は `FD001` に `connectedTo` を追加し、`FD002` の identifier を1つに直して再検証します。

In [9]:
fixed_data_ttl = '''
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:FD001 rdf:type ex:FireDoor ;
         ex:hasIdentifier "FD001" ;
         ex:connectedTo ex:ZoneA .

ex:FD002 rdf:type ex:FireDoor ;
         ex:hasIdentifier "FD002" ;
         ex:connectedTo ex:ZoneA .

ex:ZoneA rdf:type ex:AlarmZone .
'''

g_fixed = Graph()
g_fixed.parse(data=fixed_data_ttl, format='turtle')

conforms_fixed, results_graph_fixed, results_text_fixed = validate(
    data_graph=g_fixed,
    shacl_graph=Graph().parse(data=shapes_ttl, format='turtle'),
    inference='rdfs',
    abort_on_first=False,
    allow_infos=True,
    allow_warnings=True,
)

print('Conforms after fix:', conforms_fixed)
print(results_text_fixed)

Conforms after fix: True
Validation Report
Conforms: True



## 8. 次にやると良い拡張

このあと試すと理解しやすい拡張は次です。

- `should` を warning レベルとして扱う
- `may` は optional として shape から外す
- `Clause` ノードを別に作り、どの規格条文由来の shape かを保持する
- validation 結果を pandas に戻して件数集計する
- CSV から RDF を自動生成して同じ shape を当てる

この流れがそのまま、ontology / graph / validation / analysis の最小パイプラインになります。